# Cobertura e qualidade do dump Argus

## Escopo e fuso horário

Os timestamps apresentados por este notebook estão em **UTC**. O BRT (`America/Sao_Paulo`) deve ser aplicado apenas na visualização, nunca como interpretação implícita dos dados de origem.

O objetivo aqui é medir cobertura, qualidade e ritmo de chegada dos dados. Por isso, a janela usada em cada gráfico é derivada da cobertura real observada no dump, e não de datas arbitrárias.

In [ ]:
from dump_explorer.charts import (
    cadence_histogram,
    collector_timeline,
    coverage_heatmap,
    missingness_bar,
)
from dump_explorer.config import load_settings
from dump_explorer.database import create_db_engine
from dump_explorer.quality import (
    load_collector_health,
    load_missingness,
    load_series_cadence,
    load_source_coverage,
    load_source_daily_coverage,
    load_table_coverage,
)
from dump_explorer.signals import list_series, load_source_distribution
import pandas as pd

settings = load_settings()
engine = create_db_engine(settings)

## Janela real dos dados

In [ ]:
s = load_source_coverage(engine)           # rápido; retorna first_ts/last_ts em UTC
if s.empty or s['first_ts'].dropna().empty:
    raise RuntimeError("source_coverage vazio — verifique conexão/importe o dump")
start_ms = int(s['first_ts'].dropna().min().value // 10**6)
end_ms = int(s['last_ts'].dropna().max().value // 10**6) + 1
print("start_ms=", start_ms, "end_ms=", end_ms)
print("start (UTC) =", pd.to_datetime(start_ms, unit='ms', utc=True))
print("end   (UTC) =", pd.to_datetime(end_ms-1, unit='ms', utc=True))
print("Janela real usada no notebook: [start_ms, end_ms)")

## 1. coverage by table

Tabelas grandes usam estimativas; não há COUNT completo nelas.

In [ ]:
table_coverage = load_table_coverage(engine)
display(table_coverage)

## 2. Cobertura por signal_bucket.source

As épocas em milissegundos são convertidas para UTC.

In [ ]:
source_coverage = load_source_coverage(engine)
display(source_coverage)

## 3. source x day heatmap

O heatmap mostra a cobertura diária por `source`. Cada célula corresponde a um dia em UTC e a cor representa quantas linhas apareceram naquele dia. Se a matriz vier vazia, a janela não cobre o período com dados ou o recorte ficou curto demais para agregação diária útil.

In [ ]:
heatmap_start_ms = start_ms
heatmap_end_ms = end_ms
source_day = load_source_daily_coverage(engine, heatmap_start_ms, heatmap_end_ms)
display(source_day)
coverage_heatmap(source_day, x="day", y="source", value="rows").show()

## 4. Cadência para uma série selecionada

Cadência mede o intervalo entre observações consecutivas da mesma identidade `source/entity/metric`. Isso ajuda a entender se a série foi coletada de forma contínua, esporádica ou em rajadas.

Os valores de `series_start_ms` e `series_end_ms` agora seguem a janela real do dump. Os identificadores `series_source`, `series_entity` e `series_metric` ainda precisam ser escolhidos a partir de uma série existente no catálogo.

In [ ]:
source_distribution = load_source_distribution(engine, start_ms, end_ms)
selected_source = source_distribution.iloc[0]["source"] if not source_distribution.empty else None
series_catalog = (
    list_series(engine, selected_source, start_ms, end_ms) if selected_source is not None else None
)

if series_catalog is not None and not series_catalog.empty:
    first_series = series_catalog.iloc[0]
    series_source = first_series["source"]
    series_entity = first_series["entity"]
    series_metric = first_series["metric"]
    series_start_ms = start_ms
    series_end_ms = end_ms
    cadence = load_series_cadence(
        engine, series_source, series_entity, series_metric, series_start_ms, series_end_ms
    )
    display(cadence)
    cadence_histogram(cadence).show()
else:
    print("Sem série disponível para calcular cadência na janela atual.")

## 5. Resumo de nulos e desconhecidos

`missingness` resume quantos valores nulos existem por coluna no catálogo escolhido. O detalhe importante é que zero, nulo e ausência de bucket não são a mesma coisa: zero é um valor observado; nulo é um valor ausente; bucket ausente é falta de linha para aquele instante.

In [ ]:
small_table = "signal_bucket"
small_columns = ["source", "entity", "metric"]
missingness = load_missingness(engine, small_table, small_columns)
display(missingness)
missingness_bar(missingness).show()

## 6. Execuções dos coletores

Essa tabela explica como os coletores se comportaram. `collector` identifica o processo lógico; `started_at` e `finished_at` são horários de execução em UTC; `duration_s` mede tempo gasto; `rows_written` mostra produção efetiva; e `status` marca se a execução terminou (`complete`) ou ficou pela metade (`incomplete`).

In [ ]:
collector_health = load_collector_health(engine)
display(collector_health)
collector_timeline(collector_health).show()

## 7. Watermark e chegadas tardias

NPAW apresenta aproximadamente **28 horas** de atraso estrutural. Por isso, o watermark precisa preservar uma janela de releitura para chegadas tardias em vez de assumir que o dado mais recente já está completo. A ausência de tickets não é um rótulo negativo confiável: pode refletir falta de abertura, coleta ou cobertura, e não ausência do incidente.